# RAG Document Chatbot
A simple Retrieval-Augmented Generation (RAG) pipeline using **OpenAI** and **ChromaDB** — no LangChain.

**Flow:**
1. Extract text from a PDF
2. Chunk the text and embed each chunk with OpenAI
3. Store embeddings in ChromaDB
4. At query time, embed the question, retrieve the top-k chunks, and pass them as context to GPT

## 1. Install dependencies

In [1]:
#%pip install openai chromadb pypdf python-dotenv --quiet

## 2. Imports & configuration

In [2]:
import os
import pypdf
import chromadb
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
assert OPENAI_API_KEY, "Set OPENAI_API_KEY in your .env file or environment"

openai_client = OpenAI(api_key=OPENAI_API_KEY)
chroma_client = chromadb.Client()  # in-memory

EMBED_MODEL = "text-embedding-3-small"
CHAT_MODEL  = "gpt-4o-mini"
CHUNK_SIZE  = 1000
CHUNK_OVERLAP = 200
TOP_K = 4

print("Clients ready.")

Clients ready.


## 3. Helper functions

In [3]:
def extract_pdf_text(pdf_path: str) -> str:
    reader = pypdf.PdfReader(pdf_path)
    return "\n\n".join(page.extract_text() or "" for page in reader.pages)


def chunk_text(text: str, chunk_size: int = CHUNK_SIZE, overlap: int = CHUNK_OVERLAP) -> list[str]:
    chunks, start = [], 0
    while start < len(text):
        chunks.append(text[start : start + chunk_size])
        start += chunk_size - overlap
    return chunks


def get_embedding(text: str) -> list[float]:
    response = openai_client.embeddings.create(model=EMBED_MODEL, input=text)
    return response.data[0].embedding


print("Helpers defined.")

Helpers defined.


## 4. Load & index a PDF
Set `PDF_PATH` to the path of your PDF file.

In [4]:
PDF_PATH = "/Users/vijendra/agentic-ai-usecases/beginner/simple-rag/1706.03762v7.pdf"  # <-- change this

# Extract and chunk
print("Extracting text...")
raw_text = extract_pdf_text(PDF_PATH)
chunks = chunk_text(raw_text)
print(f"Pages extracted. Created {len(chunks)} chunks.")

# Build ChromaDB collection
COLLECTION_NAME = "rag_docs"
try:
    chroma_client.delete_collection(COLLECTION_NAME)
except Exception:
    pass
collection = chroma_client.create_collection(COLLECTION_NAME)

# Embed and store in batches
BATCH_SIZE = 50
for i in range(0, len(chunks), BATCH_SIZE):
    batch = chunks[i : i + BATCH_SIZE]
    embeddings = [get_embedding(c) for c in batch]
    ids = [f"chunk_{i + j}" for j in range(len(batch))]
    collection.add(embeddings=embeddings, documents=batch, ids=ids)
    print(f"  Indexed chunks {i} – {i + len(batch) - 1}")

print(f"\nDone! {collection.count()} chunks stored in ChromaDB.")

Extracting text...
Pages extracted. Created 50 chunks.
  Indexed chunks 0 – 49

Done! 50 chunks stored in ChromaDB.


## 5. Ask a single question

In [9]:

def route_message(message, openai_client):
    print(f"\n[ROUTER] Input message: {message!r}")
    response = openai_client.chat.completions.create(
        model="gpt-4o-mini",
        temperature=0,
        messages=[
            {
                "role": "system",
                "content": (
                    "You are a routing assistant. Classify the user message into one of three categories:\n"
                    "- 'chitchat': greetings, small talk, compliments, farewells, or anything unrelated to a document (e.g. 'Hi', 'How are you?', 'Thanks', 'Bye').\n"
                    "- 'overall': requests for a summary, overview, or general description of the entire document (e.g. 'summarize', 'what is this document about', 'give me an overview', 'tldr').\n"
                    "- 'rag': specific questions or requests that require looking up particular information from a document.\n"
                    "Reply with ONLY one word: chitchat, overall, or rag."
                )
            },
            {"role": "user", "content": message}
        ]
    )
    route = response.choices[0].message.content.strip().lower()
    route = route if route in ("chitchat", "overall", "rag") else "rag"
    print(f"[ROUTER] Decision: {route}")
    return route


def handle_chitchat(message, openai_client):
    print(f"\n[CHITCHAT] Handling casual message: {message!r}")
    response = openai_client.chat.completions.create(
        model="gpt-4o-mini",
        temperature=0,
        messages=[
            {
                "role": "system",
                "content": (
                    "You are a friendly assistant embedded in a PDF chatbot. "
                    "Respond naturally to casual conversation. "
                    "Keep replies short and friendly, and gently remind the user "
                    "they can upload a PDF and ask questions about it."
                )
            },
            {"role": "user", "content": message}
        ]
    )
    reply = response.choices[0].message.content
    print(f"[CHITCHAT] Response: {reply!r}")
    return reply


def summarize_document(collection, openai_client):
    print("\n[SUMMARY] Fetching first 20 chunks for document summary...")
    result = collection.get(limit=20, include=["documents"])
    chunks = result["documents"]
    print(f"[SUMMARY] Retrieved {len(chunks)} chunks")
    for i, chunk in enumerate(chunks):
        print(f"[SUMMARY] Chunk {i + 1} ({len(chunk)} chars): {chunk[:80].replace(chr(10), ' ')!r}...")

    context = "\n\n".join(chunks)
    print(f"[SUMMARY] Total context length: {len(context)} chars")
    print("[SUMMARY] Calling LLM for summary...")

    response = openai_client.chat.completions.create(
        model="gpt-4o-mini",
        temperature=0,
        messages=[
            {
                "role": "system",
                "content": (
                    "You are a helpful assistant. Using the document excerpts provided, "
                    "write a clear and concise summary of the document. "
                    "Cover the main topics, key points, and overall purpose. "
                    "Structure the summary with a brief intro, key highlights as bullet points, "
                    "and a one-sentence conclusion."
                )
            },
            {
                "role": "user",
                "content": f"Document excerpts (first 20 chunks):\n\n{context}\n\nPlease summarize this document."
            }
        ]
    )
    summary = response.choices[0].message.content
    print(f"[SUMMARY] Summary generated ({len(summary)} chars)")
    return summary


def ask_question(question, collection, openai_client):
    print(f"\n[RAG] Question: {question!r}")
    print("[RAG] Generating query embedding...")
    query_embedding = get_embedding(question)
    print("[RAG] Querying ChromaDB for top 4 relevant chunks...")
    results = collection.query(query_embeddings=[query_embedding], n_results=4)
    retrieved_chunks = results["documents"][0]
    print(f"[RAG] Retrieved {len(retrieved_chunks)} chunks:")
    for i, chunk in enumerate(retrieved_chunks):
        print(f"[RAG]   Chunk {i + 1} ({len(chunk)} chars): {chunk[:80].replace(chr(10), ' ')!r}...")
    context = "\n\n".join(retrieved_chunks)
    print(f"[RAG] Total context length: {len(context)} chars")
    print("[RAG] Calling LLM for answer...")

    response = openai_client.chat.completions.create(
        model="gpt-4o-mini",
        temperature=0,
        messages=[
            {
                "role": "system",
                "content": (
                    "You are a helpful assistant that answers questions based on the provided documents. "
                    "If you cannot find the answer in the context, say "
                    "'I cannot find this information in the provided documents.'"
                )
            },
            {
                "role": "user",
                "content": f"Context:\n{context}\n\nQuestion: {question}"
            }
        ]
    )
    answer = response.choices[0].message.content
    print(f"[RAG] Answer: {answer!r}")
    return answer

In [11]:
def chat_pipeline(message):
    """Router: directs message to chitchat handler or RAG pipeline."""
    route = route_message(message, openai_client)
    print(f"[router → {route}]")
    if route == "chitchat":
        return handle_chitchat(message, openai_client)
    elif route == "overall":
        return summarize_document(collection, openai_client)
    
    return ask_question(message,collection, openai_client)


# Quick smoke test
for msg in ["Hi there!","can you please summarize the document", "What is the main contribution of this paper?"]:
    print(f"\nQ: {msg}")
    print(f"A: {chat_pipeline(msg)}")


Q: Hi there!

[ROUTER] Input message: 'Hi there!'
[ROUTER] Decision: chitchat
[router → chitchat]

[CHITCHAT] Handling casual message: 'Hi there!'
[CHITCHAT] Response: 'Hi! How’s it going? If you have a PDF you’d like to chat about, feel free to upload it!'
A: Hi! How’s it going? If you have a PDF you’d like to chat about, feel free to upload it!

Q: can you please summarize the document

[ROUTER] Input message: 'can you please summarize the document'
[ROUTER] Decision: overall
[router → overall]

[SUMMARY] Fetching first 20 chunks for document summary...
[SUMMARY] Retrieved 20 chunks
[SUMMARY] Chunk 1 (1000 chars): 'Provided proper attribution is provided, Google hereby grants permission to repr'...
[SUMMARY] Chunk 2 (1000 chars): 'oder through an attention mechanism. We propose a new simple network architectur'...
[SUMMARY] Chunk 3 (1000 chars): 'r generalizes well to other tasks by applying it successfully to English constit'...
[SUMMARY] Chunk 4 (1000 chars): 'd efficient inferenc

## 6. Interactive chat loop
Run this cell and type your questions. Enter `quit` to stop.

In [21]:
chat_history = []

print("Chat started. Type 'quit' to exit.\n")
while True:
    user_input = input("You: ").strip()
    if user_input.lower() in ("quit", "exit", ""):
        print("Goodbye!")
        break
    print("############")
    print("User: ", user_input)
    response = chat_pipeline(user_input)
    chat_history.append({"role": "user", "content": user_input})
    chat_history.append({"role": "assistant", "content": response})

    print(f"\nAssistant: {response}\n")

Chat started. Type 'quit' to exit.

############
User:  Hi
[router → chitchat]

Assistant: Hi there! How's your day going? Remember, if you have a PDF you want to discuss, feel free to upload it!

############
User:  What is encoder in this paper?
[router → rag]

Assistant: The encoder in this paper is composed of a stack of N=6 identical layers. Each layer has two sub-layers: the first is a multi-head self-attention mechanism, and the second is a position-wise fully connected feed-forward network. The encoder employs residual connections around each of the two sub-layers, followed by layer normalization. The output of each sub-layer is computed as LayerNorm(x + Sublayer(x)), where Sublayer(x) is the function implemented by the sub-layer itself. All sub-layers in the model, as well as the embedding layers, produce outputs of dimension d_model = 512.

Goodbye!


## 7. Inspect retrieved chunks (debug)

In [ ]:
debug_question = "What is this document about?"  # <-- change this

query_embedding = get_embedding(debug_question)
results = collection.query(query_embeddings=[query_embedding], n_results=TOP_K)

for i, (doc, dist) in enumerate(zip(results["documents"][0], results["distances"][0])):
    print(f"--- Chunk {i+1} (distance: {dist:.4f}) ---")
    print(doc[:300], "...\n")